In [1]:
import os
import re
import time
import math
import json
import hashlib
from typing import List, Dict, Any, Optional, Literal, TypedDict
from dataclasses import dataclass, field
import pandas as pd
from pydantic import BaseModel, Field

from langchain.tools import tool
from langchain.chat_models import init_chat_model
from langchain_core.documents import Document
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.prompts import AIMessagePromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import WebBaseLoader, TextLoader, PyPDFLoader
from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_experimental.text_splitter import SemanticChunker

from langgraph.graph import START, END, StateGraph
from langgraph.prebuilt import ToolNode

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
from dotenv import load_dotenv

In [3]:
load_dotenv()

True

In [4]:
RAG_RESPONSE_LLM = "gpt-4o-mini"
GRADER_LLM = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

In [5]:
WEB_URLS = []
LOCAL_FILES = []

In [6]:
MAX_REWRITE_TRIES = 2
MAX_CONTEXT_CHARS = 11000
MAX_QUERY_CHARS = 11000
MIN_QUERY_CHARS = 100
MAX_RETIEVED_CHUNKS = 4
MIN_RETRIVED_CHUNKS = 2
MIN_CONTEXT_RELEVENCY = 0.9
MAX_RESPONSE_SENTENCES = 6

EMBEDDING_COST_PER_1M_TOKENS = 0.02
INPUT_TOKENS_COST_PER_1M_TOKENS = 2.00
OUTPUT_TOKENS_COST_PER_1M_TOKENS = 8.00

In [7]:
@dataclass
class TokenUsage:

    input_tokens: int = 0
    output_tokens: int = 0
    total_tokens: int = 0

In [ ]:
@dataclass
class RunTimeMetrics:

    request_id: str
    start_time: float = field(default_factory=time.time)
    end_time: float = 0.0
    retrieval_count: int = 0
    relevant_retrieval_count: int = 0
    query_rewrite_count: int = 0
    tool_calls: int = 0
    trajectory: List[str] = field(default_factory=list)
    llm_calls: int = 0
    grader_calls: int = 0
    token_usage: TokenUsage = field(default_factory=TokenUsage)
    input_guardrail_trigerred: bool = False
    injection_detected: bool = False
    response_refusal: bool = False
    is_hallucination : bool = False
    final_factualness_score: float = 0.0
    final_retrieval_relevance_score: float = 0.0
    final_answer_relevance_score: float = 0.0
    final_confidence_score: float = 0.0
    error: Optional[str] = None

    @property
    def latency_in_seconds(self) -> float:
        end = self.end_time if self.end_time else time.time()
        return end - self.start_time
    
    @property
    def estimated_cost_usd(self) -> float:
        input_cost = (self.token_usage.input_tokens / 1000000) * INPUT_TOKENS_COST_PER_1M_TOKENS
        output_cost = (self.token_usage.output_tokens / 1000000) * OUTPUT_TOKENS_COST_PER_1M_TOKENS
        
        return input_cost + output_cost

In [32]:
@dataclass
class AgentState:

    rewritten_queries: List[HumanMessagePromptTemplate] = field(default_factory=List)
    retrieved_documents: List[Document] = field(default_factory=List)
    relevant_documents: List[Document] = field(default_factory=List)
    generated_responses: List[str] = field(default_factory=List)
    retrieval_required: bool = True
    rewrite_count: int = 0
    metrics: RunTimeMetrics = field(default_factory=RunTimeMetrics)
    stop_reason: List[str] = field(default_factory=List)

In [12]:
def create_request_id(query:str) -> str:
    return hashlib.md5(f"{query}-{time.time()}").encode().hexdigest()

In [13]:
def clip_query(query:str, max_chars: int) -> str:
    return query[:max_chars] if len(query) > max_chars else query

In [14]:
def normalize_whitespace(query: str) -> str:
    return re.sub(r"\s+"," ",query).strip()

In [15]:
def calculate_tokens_per_query(query: str) -> int:
    return max(1, len(query)//4)

In [17]:
def get_tokens_usage(query: str, response: str) -> TokenUsage:

    input_tokens_usage = getattr(query, "query_metadata",None) or {}
    output_tokens_usage = getattr(response,"response_metadata",None) or {}

    return TokenUsage(input_tokens=input_tokens_usage,
                      output_tokens=output_tokens_usage)

In [18]:
def compute_total_tokens_comsumption(metrics: RunTimeMetrics, tokens_usage: TokenUsage):

    metrics.token_usage.input_tokens += tokens_usage.input_tokens
    metrics.token_usage.output_tokens += tokens_usage.output_tokens
    metrics.token_usage.total_tokens += tokens_usage.total_tokens

In [19]:
def remove_duplicated_retrieved_chunks(chunks: List[Document]) -> List[Document]:

    unique_chunks = set()

    for single_document in chunks:

        if single_document not in unique_chunks:
            unique_chunks.add(single_document)

    return unique_chunks

In [20]:
INJECTION_PATTERNS = [
    r"ignore (all|previous|prior) instructions",
    r"system prompt",
    r"reveal prompt",
    r"do not use the provided context",
    r"pretend to be",
    r"jailbreak",
    r"bypass",
    r"override safety",
    r"tool schema",
    r"prompt templates"
]

In [22]:
def validate_query(query: str) -> Dict[str,Any]:

    cleaned_query = normalize_whitespace(query)

    if len(cleaned_query) < MIN_QUERY_CHARS:
        return {"OK": False, "Reason": "Query too short"}
    
    if len(cleaned_query) > MAX_QUERY_CHARS:
        return {"OK": False, "Reason": "Query too long"}
    
    cleaned_query = cleaned_query.lower()
    is_prompt_injection = any(re.search(pattern, cleaned_query) for pattern in INJECTION_PATTERNS)

    return {"OK": True, "normalized query": cleaned_query, "prompt injection detected": is_prompt_injection}

In [23]:
def load_documents_from_files(file_paths: List[str]) -> List[Document]:

    docs = list()
    for single_file_path in file_paths:

        if not os.path.exists(single_file_path):
            continue

        normalized_path = single_file_path.lower()

        if normalized_path.endswith(".txt"):
            docs.extend(TextLoader(normalized_path,encoding="utf-8").load())
        
        elif normalized_path.endswith(".pdf"):
            docs.extend(PyPDFLoader(normalized_path).load())

        elif normalized_path.endswith(".md"):
            docs.extend(UnstructuredMarkdownLoader(normalized_path).load())

    return docs

In [26]:
def load_documents_from_urls(web_urls: List[str]) -> List[Document]:

    docs = list()
    for single_url in web_urls:
        docs.extend(WebBaseLoader(single_url).load())

    return docs

In [27]:
def load_all_documents() -> List[Document]:

    all_docs = list()

    all_docs.extend(load_documents_from_urls(WEB_URLS))
    all_docs.extend(load_documents_from_files(LOCAL_FILES))

    return all_docs

In [29]:
def build_retriever(all_documents: List[Document]):

    semantic_chunker = SemanticChunker(embeddings=OpenAIEmbeddings(model=EMBEDDING_MODEL),
                                       breakpoint_threshold_type="gradient",
                                       breakpoint_threshold_amount=0.7,
                                       min_chunk_size=50)
    
    chunk_documents = semantic_chunker.split_documents(all_documents)

    vector_store = InMemoryVectorStore.from_documents(documents=chunk_documents,
                                                      embedding=OpenAIEmbeddings(model=EMBEDDING_MODEL))
    
    retriever = vector_store.as_retriever()

    return retriever, chunk_documents

In [ ]:
all_documents = load_all_documents()
retriever, chunk_documents = build_retriever(all_documents)

In [31]:
@tool
def retrieve_context(query: str) -> str:

    """The Function retrieves the nearest chunks from the database and converts them into
        string as retrieved context which will be injected into the prompt template"""

    retrieved_chunks = retriever.invoke(query)
    de_duplicated_chunks = remove_duplicated_retrieved_chunks(retrieved_chunks)

    if not de_duplicated_chunks:
        return "NO CONTEXT FOUND"
    
    chunk_text = list()
    
    for single_chunk in de_duplicated_chunks:
        chunk_text.append(single_chunk.page_content)

    return " ".join(chunk_text)

In [33]:
def input_guardrail_node(current_agent_state: AgentState) -> AgentState:

    metrics = current_agent_state.metrics
    query = current_agent_state.rewritten_queries[-1]

    query_validation_result = validate_query(query)

    if not query_validation_result["OK"]:
        metrics.input_guardrail_trigerred = True
        current_agent_state.stop_reason = query_validation_result["reason"]
        current_agent_state.generated_responses.append(f"Query Blocked: (query_validation_result['reason'])")

        return current_agent_state
    
    else:
        current_agent_state.rewritten_queries.append(query_validation_result["normalized_query"])
        
        if query_validation_result["prompt injection detected"]:
            current_agent_state.metrics.input_guardrail_trigerred = True

    return current_agent_state